# 2.2. Подбор источников данных

**Цель этапа:** подобрать открытые источники данных по требованиям п.3 методических указаний:
- не менее 1 реляционного набора данных;
- не менее 1 иерархического набора данных (XML);
- не менее 1 источника с API (JSON);
- не менее 1 источника неструктурированных данных (тексты).

**Требования к каждому источнику:**
- ≥3 категориальных аспекта;
- ≥3 числовых характеристики;
- желательно — характеристика времени.

**Выбранные источники:**
1. UCI Online Retail II — реляционный (XLSX)
2. ЕМИСС/Росстат — иерархический (XML/SDMX)
3. Open Food Facts — API (JSON)
4. Amazon Reviews — неструктурированный (CSV, тексты)

## Источник 1. UCI Online Retail II (реляционный)

**Тип:** реляционный набор данных
**Формат:** XLSX (оригинал)
**Источник:** UCI Machine Learning Repository
**Ссылка:** https://archive.ics.uci.edu/dataset/502/online+retail+ii
**Период:** 01.12.2010 — 09.12.2011
**Объём:** 541 910 записей, 8 колонок

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| Invoice | Номинальная (ID) | Уникальный номер заказа, биекция |
| StockCode | Номинальная (ID) | Код товара, высокая уникальность |
| Description | Контентная | Текстовое описание, слабоструктурированные данные |
| Quantity | Абсолютная | Счётная величина, фиксированная единица |
| InvoiceDate | Интервальная | Дата и время, осмыслен шаг |
| Price | Отношений | Естественный ноль, допустима смена валюты |
| Customer ID | Номинальная (ID) | Идентификатор клиента, высокая уникальность |
| Country | Категориальная | Конечное множество категорий без порядка |

**Аналитическая роль:** основа для анализа потребительских предпочтений и поведенческих паттернов во времени.

In [75]:
# Импорт библиотек
import pandas as pd
import numpy as np
import requests
import time
import xml.etree.ElementTree as ET

# Настройки отображения DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [76]:
# Загрузка реляционного источника (XLSX)
start = time.time()

# Оригинал в формате XLSX
df_retail = pd.read_excel(
    'online_retail_II.xlsx',
    sheet_name='Year 2010-2011',
    engine='openpyxl'
)

load_time_retail = time.time() - start

print(f"Время загрузки: {load_time_retail:.2f} сек")
print(f"Размер: {df_retail.shape}")
print(f"Колонки: {df_retail.columns.tolist()}")
print(f"\nПервые 5 строк:")
print(df_retail.head())
print(f"\nТипы данных:")
print(df_retail.dtypes)
print(f"\nПропущенные значения:")
print(df_retail.isnull().sum())
print("Общая информация")
df_retail.info()
print("Описательная статистика")
print(df_retail.describe())

Время загрузки: 29.95 сек
Размер: (541910, 8)
Колонки: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Первые 5 строк:
  Invoice StockCode                          Description  Quantity         InvoiceDate  Price  Customer ID         Country
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00   2.55      17850.0  United Kingdom
1  536365     71053                  WHITE METAL LANTERN         6 2010-12-01 08:26:00   3.39      17850.0  United Kingdom
2  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00   2.75      17850.0  United Kingdom
3  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-12-01 08:26:00   3.39      17850.0  United Kingdom
4  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6 2010-12-01 08:26:00   3.39      17850.0  United Kingdom

Типы данных:
Invoice                object
StockCode              object
Description   

In [77]:
# Категориальные аспекты
print("Категориальные аспекты:")
print(f"  Country: {df_retail['Country'].nunique()} уникальных")
print(f"  StockCode: {df_retail['StockCode'].nunique()} уникальных")
print(f"  Description: {df_retail['Description'].nunique()} уникальных")

# Числовые характеристики
print("\nЧисловые характеристики:")
print(f"  Quantity: min={df_retail['Quantity'].min()}, max={df_retail['Quantity'].max()}")
print(f"  Price: min={df_retail['Price'].min()}, max={df_retail['Price'].max()}")
print(f"  Customer ID: {df_retail['Customer ID'].nunique()} уникальных")

# Временная характеристика
print("\nВременная характеристика:")
print(f"  InvoiceDate: {df_retail['InvoiceDate'].min()} — {df_retail['InvoiceDate'].max()}")

Категориальные аспекты:
  Country: 38 уникальных
  StockCode: 4070 уникальных
  Description: 4223 уникальных

Числовые характеристики:
  Quantity: min=-80995, max=80995
  Price: min=-11062.06, max=38970.0
  Customer ID: 4372 уникальных

Временная характеристика:
  InvoiceDate: 2010-12-01 08:26:00 — 2011-12-09 12:50:00


In [78]:
# Проверка дубликатов
duplicates = df_retail.duplicated().sum()
print(f"Дубликатов в Retail: {duplicates}")

Дубликатов в Retail: 5268


**Вывод:** источник загружен успешно. Содержит 3 категориальных аспекта (Country, StockCode, Description) и 3 числовых характеристики (Quantity, Price, Customer ID). Присутствует временная характеристика (InvoiceDate). Источник уникален.

## Источник 2. ЕМИСС/Росстат (иерархический, XML)

**Тип:** иерархический набор данных
**Формат:** XML (SDMX)
**Источник:** Единая межведомственная информационно-статистическая система (fedstat.ru)
**Показатель:** Оборот розничной торговли пищевыми продуктами, включая напитки, и табачными изделиями, проданных на розничных рынках и ярмарках (ID 43271)
**Период:** 2022 год
**Объём:** 188 записей

**Структура XML:**
- CodeLists → справочник ОКАТО (103 региона)
- Description → метаданные показателя
- DataSet → серии наблюдений

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| okato | Номинальная (ID) | Код региона, биекция |
| region | Категориальная | Название региона, конечное множество |
| period | Категориальная | Период (январь-июнь / январь-декабрь) |
| year | Интервальная | Год, осмыслен шаг |
| value | Отношений | Значение оборота, естественный ноль |
| unit | Категориальная | Единица измерения |

**Аналитическая роль:** данные о розничном товарообороте по регионам РФ — фактор внешней среды.

In [79]:
# Парсинг XML (ЕМИСС/Росстат)
start = time.time()

tree = ET.parse('data.xml')
root = tree.getroot()

# Пространства имён SDMX
ns = {
    'generic': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/generic',
    'structure': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/structure',
    'common': 'http://www.SDMX.org/resources/SDMXML/schemas/v1_0/common'
}

print(f"Корневой тег: {root.tag}")
print(f"Время парсинга XML: {time.time() - start:.2f} сек")

Корневой тег: {http://www.SDMX.org/resources/SDMXML/schemas/v1_0/message}GenericData
Время парсинга XML: 0.06 сек


In [80]:
# Извлечение справочника регионов (ОКАТО)
okato_dict = {}

codelist = root.find('.//structure:CodeList[@id="s_OKATO"]', ns)
if codelist is not None:
    for code in codelist.findall('structure:Code', ns):
        value = code.get('value')
        desc = code.find('structure:Description', ns)
        if desc is not None:
            okato_dict[value] = desc.text

print(f"Регионов в справочнике: {len(okato_dict)}")
print(f"\nПримеры:")
for k, v in list(okato_dict.items())[:5]:
    print(f"  {k}: {v}")

Регионов в справочнике: 103

Примеры:
  643: Российская Федерация
  643004.АГ: Российская Федерация без учета новых субъектов (с 01.01.2023)
  030: Центральный федеральный округ
  14000000000: Белгородская область
  15000000000: Брянская область


In [81]:
# Извлечение данных из XML
records = []

for series in root.findall('.//generic:Series', ns):
    # Ключ серии (регион)
    series_key = series.find('generic:SeriesKey', ns)
    okato_code = None
    if series_key is not None:
        value_elem = series_key.find('generic:Value', ns)
        if value_elem is not None:
            okato_code = value_elem.get('value')
    
    # Атрибуты (единица измерения, период)
    period = None
    unit = None
    attrs = series.find('generic:Attributes', ns)
    if attrs is not None:
        for val in attrs.findall('generic:Value', ns):
            if val.get('concept') == 'PERIOD':
                period = val.get('value')
            elif val.get('concept') == 'EI':
                unit = val.get('value')
    
    # Наблюдения (год + значение)
    for obs in series.findall('generic:Obs', ns):
        time_elem = obs.find('generic:Time', ns)
        value_elem = obs.find('generic:ObsValue', ns)
        
        records.append({
            'okato': okato_code,
            'region': okato_dict.get(okato_code, 'Неизвестно'),
            'period': period,
            'year': time_elem.text if time_elem is not None else None,
            'value': float(value_elem.get('value').replace(',', '.')) if value_elem is not None else None,
            'unit': unit
        })

df_xml = pd.DataFrame(records)

print(f"Извлечено записей: {len(df_xml)}")
print(f"Колонки: {df_xml.columns.tolist()}")
print(df_xml.head(10))

Извлечено записей: 188
Колонки: ['okato', 'region', 'period', 'year', 'value', 'unit']
         okato                         region          period  year         value           unit
0          643           Российская Федерация     январь-июнь  2022  4.620429e+08  тысяча рублей
1          643           Российская Федерация  январь-декабрь  2022  1.016340e+09  тысяча рублей
2          030  Центральный федеральный округ     январь-июнь  2022  1.329859e+08  тысяча рублей
3          030  Центральный федеральный округ  январь-декабрь  2022  2.837889e+08  тысяча рублей
4  14000000000           Белгородская область     январь-июнь  2022  6.178408e+06  тысяча рублей
5  14000000000           Белгородская область  январь-декабрь  2022  1.323196e+07  тысяча рублей
6  15000000000               Брянская область     январь-июнь  2022  5.848131e+06  тысяча рублей
7  15000000000               Брянская область  январь-декабрь  2022  1.161781e+07  тысяча рублей
8  17000000000           Владимирская об

In [82]:
# Характеристики XML-источника
print("Характеристики XML-Источника (ЕМИСС/Росстат)")

print("\nКатегориальные аспекты (>=3):")
print(f"  1. region: {df_xml['region'].nunique()} уникальных")
print(f"  2. period: {df_xml['period'].nunique()} уникальных ({df_xml['period'].unique().tolist()})")
print(f"  3. year: {df_xml['year'].nunique()} уникальных")
print(f"  4. unit: {df_xml['unit'].nunique()} уникальных")

print("\nЧисловые характеристики (>=3):")
print(f"  1. value: min={df_xml['value'].min()}, max={df_xml['value'].max()}, mean={df_xml['value'].mean():.2f}")
print(f"  2. okato (код): {df_xml['okato'].nunique()} уникальных")
print(f"  3. Наблюдений на регион: {df_xml.groupby('region').size().mean():.1f}")

print("\nВременная характеристика:")
print(f"  year: {df_xml['year'].min()} — {df_xml['year'].max()}")

print(f"\nОбщий размер: {df_xml.shape}")

# Общая информация
print("\nОбщая информация:")
df_xml.info()

# Описательная статистика
print("\nОписательная статистика:")
print(df_xml.describe())

Характеристики XML-Источника (ЕМИСС/Росстат)

Категориальные аспекты (>=3):
  1. region: 94 уникальных
  2. period: 2 уникальных (['январь-июнь', 'январь-декабрь'])
  3. year: 1 уникальных
  4. unit: 1 уникальных

Числовые характеристики (>=3):
  1. value: min=0.0, max=1016340361.7, mean=22992553.75
  2. okato (код): 94 уникальных
  3. Наблюдений на регион: 2.0

Временная характеристика:
  year: 2022 — 2022

Общий размер: (188, 6)

Общая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 188 entries, 0 to 187
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   okato   188 non-null    object 
 1   region  188 non-null    object 
 2   period  188 non-null    object 
 3   year    188 non-null    object 
 4   value   188 non-null    float64
 5   unit    188 non-null    object 
dtypes: float64(1), object(5)
memory usage: 8.9+ KB

Описательная статистика):
              value
count  1.880000e+02
mean   2.299255e+07
std    

**Вывод:** источник загружен успешно. Содержит 4 категориальных аспекта (region, period, year, unit) и 3 числовых характеристики (value, okato, количество наблюдений). Присутствует временная характеристика (year). Источник уникален, не пересекается с Retail.

## Источник 3. Open Food Facts (API, JSON)

**Тип:** API с выдачей в формате JSON
**Источник:** Open Food Facts
**Ссылка:** https://world.openfoodfacts.org/api/v2/search
**Аутентификация:** не требуется
**Объём:** 100 товаров (категория «snacks»)

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| code | Номинальная (ID) | Штрихкод товара, биекция |
| product_name | Контентная | Название товара, слабоструктурированный текст |
| brands | Категориальная | Бренд, конечное множество |
| categories | Категориальная | Категории товаров |
| countries | Категориальная | Страны продажи |
| nutriscore_grade | Порядковая | Оценка a-e, упорядоченные градации без равного интервала |
| popularity_key | Отношений | Популярность, естественный ноль |
| last_modified_t | Интервальная | Дата изменения, осмыслен шаг |

**Аналитическая роль:** информация о товарах и их характеристиках — дополняет анализ потребительских предпочтений.

In [83]:
# Загрузка API (Open Food Facts)
start = time.time()

url = "https://world.openfoodfacts.org/api/v2/search"
params = {
    'categories': 'snacks',
    'fields': 'code,product_name,brands,categories,countries,nutriscore_grade,popularity_key,last_modified_t',
    'page_size': 100
}

headers = {'User-Agent': 'StudentResearch/1.0'}
response = requests.get(url, params=params, headers=headers, timeout=30)
data = response.json()

load_time_api = time.time() - start

print(f"Время загрузки API: {load_time_api:.2f} сек")
print(f"Статус: {response.status_code}")
print(f"Товаров: {len(data.get('products', []))}")

df_api = pd.json_normalize(data['products'])
print(f"Колонки: {df_api.columns.tolist()}")
print(df_api.head())

print("Общая информация")
df_api.info()

print("Описательная статистика")
print(df_api.describe())

Время загрузки API: 1.73 сек
Статус: 200
Товаров: 100
Колонки: ['brands', 'categories', 'code', 'countries', 'last_modified_t', 'nutriscore_grade', 'popularity_key', 'product_name']
                    brands                                         categories           code                                          countries  last_modified_t nutriscore_grade  popularity_key  \
0  Milky Food Professional                Cheeses, fr:Fromages blancs natures  6111246721261             Canada, France, Morocco, United States       1788024732                c     24999993419   
1                 سيدي علي      Unsweetened beverages, Natural mineral waters  6111035000430  Canada, France, Morocco, United Kingdom, Unite...       1790034151                a     24999992968   
2                   Jaouda  Snacks, Plain fermented dairy desserts with cream  6111242100992  France, India, Morocco, Netherlands, United St...       1785929247                a     24999992470   
3                 sidi ali    

In [84]:
print("Категориальные аспекты:")
if 'brands' in df_api.columns:
    print(f"  brands: {df_api['brands'].nunique()} уникальных")
if 'categories' in df_api.columns:
    print(f"  categories: {df_api['categories'].nunique()} уникальных")
if 'countries' in df_api.columns:
    print(f"  countries: {df_api['countries'].nunique()} уникальных")

print("\nЧисловые характеристики:")
if 'popularity_key' in df_api.columns:
    print(f"  popularity_key: min={df_api['popularity_key'].min()}, max={df_api['popularity_key'].max()}")
if 'code' in df_api.columns:
    print(f"  code: {df_api['code'].nunique()} уникальных")

print("\nВременная характеристика:")
if 'last_modified_t' in df_api.columns:
    df_api['last_modified_date'] = pd.to_datetime(df_api['last_modified_t'], unit='s')
    print(f"  last_modified: {df_api['last_modified_date'].min()} — {df_api['last_modified_date'].max()}")

Категориальные аспекты:
  brands: 76 уникальных
  categories: 85 уникальных
  countries: 55 уникальных

Числовые характеристики:
  popularity_key: min=24999500702, max=24999993419
  code: 100 уникальных

Временная характеристика:
  last_modified: 2026-04-03 16:19:58 — 2026-09-28 06:22:02


**Вывод:** источник загружен успешно. Содержит 4 категориальных аспекта (brands, categories, countries, nutriscore_grade) и 3 числовых характеристики (popularity_key, code, количество). Присутствует временная характеристика (last_modified). Источник уникален.

## Источник 4. Amazon Reviews (неструктурированный, CSV)

**Тип:** неструктурированные данные (тексты)
**Формат:** CSV
**Источник:** Kaggle
**Ссылка:** https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
**Объём:** 50 000 записей (из 568 454)

**Поля и аналитические шкалы:**

| Поле | Аналитическая шкала | Обоснование |
|------|---------------------|-------------|
| Id | Номинальная (ID) | Идентификатор записи, биекция |
| ProductId | Номинальная (ID) | Идентификатор товара |
| UserId | Номинальная (ID) | Идентификатор пользователя |
| ProfileName | Контентная | Имя пользователя, текст |
| HelpfulnessNumerator | Абсолютная | Счётная величина, фиксированная единица |
| HelpfulnessDenominator | Абсолютная | Счётная величина, фиксированная единица |
| Score | Порядковая | Оценка 1-5, упорядоченные градации |
| Time | Интервальная | Время отзыва, осмыслен шаг |
| Summary | Контентная | Краткое резюме, текст |
| Text | Контентная | Текст отзыва, слабоструктурированные данные |

**Аналитическая роль:** тексты отзывов для анализа потребительских предпочтений и тональности.

In [85]:
# Загрузка неструктурированного источника (Amazon Reviews)
start = time.time()

df_reviews = pd.read_csv('Reviews.csv', nrows=50000)

load_time_reviews = time.time() - start

print(f"Время загрузки: {load_time_reviews:.2f} сек")
print(f"Размер: {df_reviews.shape}")
print(f"Колонки: {df_reviews.columns.tolist()}")
print(df_reviews.head())

print("Общая информация")
df_reviews.info()

print("Описательная статистика")
print(df_reviews.describe())

Время загрузки: 0.21 сек
Размер: (50000, 10)
Колонки: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']
   Id   ProductId          UserId                      ProfileName  HelpfulnessNumerator  HelpfulnessDenominator  Score        Time                Summary  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian                     1                       1      5  1303862400  Good Quality Dog Food   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa                     0                       0      1  1346976000      Not as Advertised   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"                     1                       1      4  1219017600  "Delight" says it all   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl                     3                       3      2  1307923200         Cough Medicine   
4   5  B006K2ZZ7K  A1UQRS

In [86]:
print("Категориальные аспекты:")
print(f"  ProductId: {df_reviews['ProductId'].nunique()} уникальных")
print(f"  UserId: {df_reviews['UserId'].nunique()} уникальных")
print(f"  ProfileName: {df_reviews['ProfileName'].nunique()} уникальных")
print(f"  Score: {df_reviews['Score'].nunique()} уникальных значений")

print("\nЧисловые характеристики:")
print(f"  Score: min={df_reviews['Score'].min()}, max={df_reviews['Score'].max()}, mean={df_reviews['Score'].mean():.2f}")
print(f"  HelpfulnessNumerator: min={df_reviews['HelpfulnessNumerator'].min()}, max={df_reviews['HelpfulnessNumerator'].max()}")
print(f"  HelpfulnessDenominator: min={df_reviews['HelpfulnessDenominator'].min()}, max={df_reviews['HelpfulnessDenominator'].max()}")

print("\nВременная характеристика:")
df_reviews['review_date'] = pd.to_datetime(df_reviews['Time'], unit='s')
print(f"  Диапазон: {df_reviews['review_date'].min()} — {df_reviews['review_date'].max()}")

print("\nНеструктурированные данные:")
df_reviews['text_length'] = df_reviews['Text'].astype(str).str.len()
df_reviews['summary_length'] = df_reviews['Summary'].astype(str).str.len()
print(f"  Text: средняя длина {df_reviews['text_length'].mean():.1f} символов")
print(f"  Summary: средняя длина {df_reviews['summary_length'].mean():.1f} символов")

print("\nПропущенные значения:")
print(df_reviews.isnull().sum())

Категориальные аспекты:
  ProductId: 6115 уникальных
  UserId: 40048 уникальных
  ProfileName: 36925 уникальных
  Score: 5 уникальных значений

Числовые характеристики:
  Score: min=1, max=5, mean=4.15
  HelpfulnessNumerator: min=0, max=398
  HelpfulnessDenominator: min=0, max=401

Временная характеристика:
  Диапазон: 2000-06-23 00:00:00 — 2012-10-26 00:00:00

Неструктурированные данные:
  Text: средняя длина 431.2 символов
  Summary: средняя длина 23.4 символов

Пропущенные значения:
Id                        0
ProductId                 0
UserId                    0
ProfileName               5
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   2
Text                      0
review_date               0
text_length               0
summary_length            0
dtype: int64


In [87]:
# Проверка пересечения стран между Retail и API
retail_countries = set(df_retail['Country'].unique())
api_countries = set()
for countries_str in df_api['countries'].dropna():
    api_countries.update([c.strip() for c in countries_str.split(',')])

common = retail_countries & api_countries
print(f"Общих стран: {len(common)}")
print(f"Примеры: {list(common)[:10]}")

Общих стран: 18
Примеры: ['Netherlands', 'Italy', 'Poland', 'Canada', 'Sweden', 'France', 'Portugal', 'Austria', 'Czech Republic', 'Greece']


**Вывод:** источник загружен успешно. Содержит 4 категориальных аспекта (ProductId, UserId, ProfileName, Score) и 3 числовых характеристики (Score, HelpfulnessNumerator, HelpfulnessDenominator). Присутствует временная характеристика (Time). Неструктурированные данные: Text, Summary. Источник уникален.

## Сводная таблица источников

| № | Источник | Тип | Формат | Записей | Катег. аспектов | Числовых | Временной диапазон |
|---|----------|-----|--------|---------|-----------------|----------|-------------------|
| 1 | UCI Online Retail II | Реляционный | XLSX | 541 910 | 3 | 3 | 01.12.2010 — 09.12.2011 |
| 2 | ЕМИСС/Росстат | Иерархический | XML | 188 | 4 | 3 | 2022 год |
| 3 | Open Food Facts | API | JSON | 100 | 4 | 3 | 03.04.2026 — 28.09.2026 |
| 4 | Amazon Reviews | Неструктурированный | CSV | 50 000 | 4 | 3 | 23.06.2000 — 26.10.2012 |

**Все источники уникальны и не пересекаются между собой.**